In [1]:
import pandas as pd

terror_df = pd.read_excel(r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\GTD_inferential.xlsx", sheet_name="count")
# or pd.read_csv("terrorism_data.csv")


In [2]:
country_map = {
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Swaziland": "Eswatini",
    "Macedonia": "North Macedonia",
    "Slovak Republic": "Slovakia",
    "East Timor": "Timor-Leste",

    # Congo
    "People's Republic of the Congo": "Congo",
    "Republic of the Congo": "Congo",
    "Zaire": "Congo",

    # Germany
    "East Germany (GDR)": "Germany",
    "West Germany (FRG)": "Germany",

    # Yemen
    "North Yemen": "Yemen",
    "South Yemen": "Yemen",

    # Historical states (drop)
    "Czechoslovakia": None,
    "Soviet Union": None,
    "Yugoslavia": None,

    # Palestine
    "West Bank and Gaza Strip": "Palestine"
}

terror_df["Country"] = terror_df["Country"].map(lambda x: country_map.get(x, x))

# Drop obsolete entities
terror_df = terror_df[terror_df["Country"].notna()]


In [3]:
year_cols = terror_df.columns.drop("Country")

terror_df = (
    terror_df
    .groupby("Country", as_index=False)[year_cols]
    .sum()
)


In [4]:
terror_long = terror_df.melt(
    id_vars="Country",
    var_name="Year",
    value_name="Attacks"
)

terror_long["Year"] = terror_long["Year"].astype(int)
terror_long["Attacks"] = terror_long["Attacks"].fillna(0).astype(int)


In [9]:
import numpy as np
terror_long["log_attacks"] = np.log1p(terror_long["Attacks"])
print(terror_long)

                Country  Year  Attacks  log_attacks
0           Afghanistan  1970        0     0.000000
1               Albania  1970        0     0.000000
2               Algeria  1970        0     0.000000
3               Andorra  1970        0     0.000000
4                Angola  1970        0     0.000000
...                 ...   ...      ...          ...
9745  Wallis and Futuna  2020        0     0.000000
9746     Western Sahara  2020        0     0.000000
9747              Yemen  2020      474     6.163315
9748             Zambia  2020        0     0.000000
9749           Zimbabwe  2020        1     0.693147

[9750 rows x 4 columns]


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from functools import reduce
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import scipy.stats as stats
from scipy.stats import jarque_bera, shapiro

import statsmodels.api as sm


In [ ]:

# Calculate % of NaN values in Attacks for each country
nan_percent = (
    terror_long
    .groupby("Country")["Attacks"]
    .apply(lambda x: x.isna().mean() * 100)
)

# Sort descending to see countries with most missing values first
nan_percent = nan_percent.sort_values(ascending=False)

print(nan_percent)


In [ ]:
# Pivot data to have countries as rows and years as columns
gdp_pivot = terror_long.pivot(index="Country Name", columns="Year", values="GDP_Growth")

# Create a boolean mask: True = missing, False = present
missing_mask = gdp_pivot.isna()

# Plot the heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(missing_mask, cmap=["#2ca02c", "#d62728"], cbar_kws={'label': 'Missing Data'}, linewidths=0.5)
plt.title("Missing GDP Growth Data per Country over Years")
plt.xlabel("Year")
plt.ylabel("Country")
plt.show()

In [ ]:
valid_countries = gdp_long.groupby("Country Name")["GDP_Growth"].apply(lambda x: x.notna().sum() > 0)
gdp_clean = gdp_long[gdp_long["Country Name"].isin(valid_countries[valid_countries].index)]
gdp_clean = gdp_clean.dropna(subset=["GDP_Growth"])
print(f"After removing countries with all NaN values, data shape: {gdp_clean.shape}")
print(gdp_clean)


In [ ]:
import numpy as np

# Check number of NaNs, inf, -inf
print("NaNs:", gdp_clean["GDP_Growth"].isna().sum())
print("Infinite values:", np.isinf(gdp_clean["GDP_Growth"]).sum())
print("Negative Infinite values:", np.isneginf(gdp_clean["GDP_Growth"]).sum())

In [ ]:

# Histogram + KDE
plt.figure(figsize=(10,6))
sns.histplot(gdp_clean["GDP_Growth"], bins=50, kde=True, color='steelblue')
plt.title("Histogram + KDE of GDP Growth % (1970–2024)")
plt.xlabel("GDP Growth (%)")
plt.ylabel("Frequency")
plt.show()

# QQ Plot
plt.figure(figsize=(6,6))
sm.qqplot(gdp_clean["GDP_Growth"], line='45', fit=True)
plt.title("QQ Plot of GDP Growth %")
plt.show()


In [ ]:
skewness = stats.skew(gdp_clean["GDP_Growth"])
kurtosis = stats.kurtosis(gdp_clean["GDP_Growth"])  # excess kurtosis
print(f"Skewness: {skewness:.4f}")
print(f"Excess Kurtosis: {kurtosis:.4f}")

In [ ]:
from scipy.stats import shapiro, kstest, anderson, norm
import numpy as np


# Kolmogorov-Smirnov Test
mu, sigma = gdp_clean["GDP_Growth"].mean(), gdp_clean["GDP_Growth"].std()
stat, p = kstest(gdp_clean["GDP_Growth"], 'norm', args=(mu, sigma))
print("\nKolmogorov-Smirnov Test")
print(f"Statistic={stat:.4f}, p-value={p:.4f}")

# Anderson-Darling Test
result = anderson(gdp_clean["GDP_Growth"], dist='norm')
print("\nAnderson-Darling Test")
print(f"Statistic={result.statistic:.4f}")
for level, crit in zip(result.significance_level, result.critical_values):
    print(f"  {level}%: {crit:.4f}")


In [ ]:
#BOXPLOT
plt.figure(figsize=(8,4))
plt.boxplot(gdp_clean["GDP_Growth"], vert=False)
plt.title("Boxplot of GDP Growth")
plt.xlabel("GDP Growth (%)")
plt.show()


In [ ]:
Q1 = np.percentile(gdp_clean["GDP_Growth"], 25)
Q3 = np.percentile(gdp_clean["GDP_Growth"], 75)
IQR = Q3 - Q1

lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR

iqr_outliers = (gdp_clean["GDP_Growth"] < lower_iqr) | (gdp_clean["GDP_Growth"] > upper_iqr)

print("IQR Outliers Count:", iqr_outliers.sum())
print("IQR Outliers %:", iqr_outliers.mean() * 100)


In [ ]:
#EXTREME QUANTILE ANALYSIS
p1, p99 = np.percentile(gdp_clean["GDP_Growth"], [1, 99])

extreme_quantiles = (gdp_clean["GDP_Growth"] < p1) | (gdp_clean["GDP_Growth"] > p99)

print("1–99% Extreme Values Count:", extreme_quantiles.sum())
print("Lower 1% cutoff:", p1)
print("Upper 99% cutoff:", p99)
